# Express Links · Week 4, step 13: the seven graph versions on PEMS-BAY and Sharjah (Mohammed)

**Goal:** the seven graph versions (original, random, map_only, corr_only, same_side, express, virtual_node) on **PEMS-BAY**, the second real benchmark (325 sensors), and on **Sharjah**, the real road layout built in step 11 (300 sensors), three seeds each: **42 runs**. Together with Al Zahraa's step 12 (toy city and METR-LA) that makes **84 runs**, each with saved forecasts and a scored results row, so every dataset listed in proposal 3.2 is covered. PEMS-BAY checks the result on a second benchmark; Sharjah checks it on a real layout.

Some words used below. A **run** is one training of Graph WaveNet on one dataset with one graph version and one **seed** (the random starting point; three seeds per version give a mean ± std). An **epoch** is one pass over the training data. GPU time is counted in **GPU-hours** on Colab's **T4** card.

**Nothing is trained differently here.** The training loop, the resume logic and the scoring hook are Al Zahraa's `code/train_gwnet.py` from step 12, imported from Drive (section 3), never rewritten. This notebook only adds the two datasets, the input checks, the time planning and the progress reports. The epoch count is the one step 12 decided (100 epochs, no early stopping, folder `runs/`; section 5 checks this against the step-12 runs in Drive).

**How long it takes.** Measured on the T4: 131 s per epoch on METR-LA (207 sensors, 24k training samples). Time per epoch grows with the number of sensors and with the number of training samples, so PEMS-BAY (325 sensors, 36k samples) takes about **5.2 min per epoch, 8.8 h per 100-epoch run**, and Sharjah (300 sensors, 24k samples) about **3.2 min per epoch, 5.4 h per run**. The 42 runs are therefore about **300 GPU-hours, 12.5 days of one T4 running day and night**; the 14 runs of seed 0 alone are about 100 GPU-hours. These are T4 numbers: an A100 is roughly 3 to 5 times faster, an L4 about 1.5 to 2 times (section 3). Section 6 recomputes everything from the real epoch times as soon as they exist, gives the ETA of **every run**, and says whether it fits before `DEADLINE` and within your Colab compute units.

**Run it in batches.** Every epoch saves `last.pt` to Drive, so nothing is ever lost except the epoch that was running (at most about 5 minutes). One run of this notebook is one batch:

1. Set `SESSION_HOURS` to the time you have now (or leave `None` to go on until Colab stops you), then **Runtime → Run all**.
2. Section 6 prints **where we are** (done / resumable / not started / blocked), the **ETA of each run**, and what **this session** will get done.
3. Section 7 trains; after every epoch it prints the time left in that run, and after every finished run the time left overall.
4. When the session ends (your limit, a disconnect, or you stop it), just **Run all** again: it continues from the last saved epoch. A run your crashed session was training is taken back automatically (section 2, `RECLAIM_MY_RUNS`).

Progress is also written to `runs/progress_step13.txt` in Drive, so you can read the ETA on your phone without opening Colab.

**Depends on**

| Needs | From | Status |
|---|---|---|
| `data/processed/PEMS-BAY/` and `data/raw/adj_mx_bay.pkl` | step 2 (Mohammed) | done |
| `data/processed/SHARJAH/`, `data/raw/sharjah/` (graph, regions, jam log) | step 11 (Mohammed) | done |
| `data/processed/PEMS-BAY/bottleneck/regions.npz` | step 5 (Mohammed) | done |
| `code/train_gwnet.py` and the step-12 decision on `EPOCHS` | **step 12 (Al Zahraa)** | in Drive since 31 Oct, updated 7 Nov; section 3 imports it |
| the **seven graph files** per dataset under `data/graphs/<dataset>/` | **step 10 (Al Zahraa)** | section 5 checks which exist; missing versions show as `blocked` and are skipped, so training starts on the versions that exist |
| `code/metrics.py` | step 9 (Mohammed) | done |

**Run it:** Runtime → Change runtime type → **GPU**, check section 2, then Runtime → Run all. Keep the tab open (or use Colab Pro background execution). When it disconnects: reconnect, Run all. On the GPU server (no Colab): sync the Coding folder, set the environment variable `EXPRESS_PROJECT_DIR` to it, and run the same cells.

## The "across a bottleneck" rule for these two datasets

Step 9's scoring reports every number twice: for all roads, and for roads **behind a bottleneck**, the ones a jam has to cross a bottleneck to reach. Where that subset comes from:

* **PEMS-BAY:** step 5's spectral districts, in `data/processed/PEMS-BAY/bottleneck/regions.npz` (`behind_bottleneck`: a fixed set of roads, so both recall and precision are reported for them).
* **Sharjah:** the lagoon. Step 11 split the city into the two sides of the Khalid / Al Khan lagoon (`data/raw/sharjah/SHARJAH_regions.json`: two roads are across the bottleneck when their districts differ) and logged, for every simulated jam, whether it **crossed** to the other side (`SHARJAH_jam_arrivals.csv`, column `crossed`). As for the toy cities, "behind the bottleneck" is then a property of the jam, not of the road, because jams start on both sides; recall is reported for those jams and precision is left empty.

Section 5 makes sure `metrics.py` can find both files. The rule itself belongs to steps 10 and 11; it is only recorded here so the report and the code agree.

## 1. Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, glob, sys, time, json, shutil, filecmp, pickle, math, re
from datetime import datetime, timedelta
import numpy as np, pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER = os.path.join('AdvML_F26_Project_&_Survey_Paper', 'Project', 'Coding')
    ROOTS = ['/content/drive/MyDrive', '/content/drive/.shortcut-targets-by-id', '/content/drive/Shareddrives']
    hits = []
    for root in ROOTS:
        for depth in range(4):
            hits += glob.glob(os.path.join(root, *(['*'] * depth), FOLDER))
    hits = sorted(set(h for h in hits if os.path.isdir(os.path.join(h, 'data'))), key=len)
    assert hits, ("Couldn't find the 'Coding' folder. In Google Drive open 'Shared with me', right-click "
                  "'AdvML_F26_Project_&_Survey_Paper' -> Organize -> Add shortcut -> My Drive, then rerun this cell.")
    PROJECT_DIR = hits[0]
else:
    # not Colab (the GPU server, or a local test): EXPRESS_PROJECT_DIR must point at a synced copy of the Coding folder
    PROJECT_DIR = os.environ.get('EXPRESS_PROJECT_DIR', '')
    assert os.path.isdir(os.path.join(PROJECT_DIR, 'data')), \
        'Not in Colab: set the environment variable EXPRESS_PROJECT_DIR to the Coding folder (synced from Drive) and restart.'
print('PROJECT_DIR =', PROJECT_DIR)
try:
    probe = os.path.join(PROJECT_DIR, '.write_test'); open(probe, 'w').close(); os.remove(probe)
    print('OK  you can write to this folder')
except OSError:
    raise PermissionError('You can read this folder but not write to it. Ask the owner to share it with you as Editor.')
CODE_DIR = os.path.join(PROJECT_DIR, 'code')

try:                                           # clock in Sharjah time (Colab's clock is UTC)
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo('Asia/Dubai')
except Exception:
    TZ = None
def now_local():
    return datetime.now(TZ) if TZ else datetime.now()
print('now:', now_local().strftime('%a %d %b %Y %H:%M'), '(Sharjah time)' if TZ else '(server time)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding
OK  you can write to this folder
now: Tue 10 Nov 2026 13:56 (Sharjah time)


## 2. Settings

* `WORKER` – `'mohammed'`. **One session per worker name.** If a second GPU joins (Al Zahraa's Colab once her 42 runs are done, the GPU server, a second account of yours), give it its own name (`'alzahraa'`, `'mohammed2'`) and, simplest, its own `ONLY_DATASET` (one takes `'PEMS-BAY'`, the other `'SHARJAH'`); the claim files stop two sessions from training the same run in any case.
* `EPOCHS`, `PATIENCE`, `RUNS_SUBDIR` – **the step-12 decision: 100 epochs, no early stopping, `runs/`** (the pilot still improved by 0.10 mph after epoch 50, so 100 epochs stay). All 84 runs of the project must share one training length, or the versions cannot be compared across datasets. `PATIENCE = None` trains every epoch; a number would stop a run once the validation error (the error on the validation split, the 10% of the data that follows the training part) has not improved for that many epochs. Section 5 compares these values with the step-12 runs already in Drive and stops if they differ.
* `SESSION_HOURS` – **one batch.** The hours you have now; the notebook stops cleanly after the epoch that crosses this limit and prints where it got to. `None` = continue until no free run is left or Colab stops you. With Colab Pro+ background execution (the session keeps running after you close the tab, up to 24 h) set it to about `23` so the stop is clean. `MAX_RUNS` and `ONLY_DATASET` narrow a session further.
* `RECLAIM_MY_RUNS` – leave `True`. After a crash of your session, the run it was training still shows `in_progress` under your name, because the module cannot tell a crashed session from a live one until the claim is 30 minutes old (a claim's **heartbeat** is a time stamp the training session refreshes every epoch). With `True`, section 6 takes back any claim that carries your worker name and is older than `OWN_STALE_MIN` (10) minutes, and waits for a younger one to reach that age first (at most 10 minutes), so the interrupted run resumes in this session instead of a new run being started.
* `GPU_HOURS_PER_DAY`, `N_GPUS`, `DEADLINE` – for the time planning in section 6 only: how many hours a day one GPU really trains (free Colab: 3 to 4; Colab Pro with background execution: up to 24), how many GPUs run this notebook at the same time (each with its own `WORKER`), and the day by which the last run must be finished.
* `COMPUTE_UNITS_LEFT`, `UNITS_PER_HOUR` – Colab Pro / Pro+ only, both read off Colab's **Resources** panel (the units left on the account, and what the current GPU costs per hour). Section 6 then says how many GPU-hours and how many runs the units buy. Leave `None` on a free account.

Model settings are the step-3 baseline settings (Graph WaveNet defaults, learned links off) and are fixed inside `train_gwnet.py`. The job list is seed-major: the 14 runs of seed 0 first, then seed 1, then seed 2, which is also the plan's first cut if time is short.

In [3]:
WORKER        = 'mohammed'
DATASETS      = ['PEMS-BAY', 'SHARJAH']      # the step-2 and step-11 dataset names (section 4 checks them)

# ---- the step-12 decision (Week3_Step12 section 2 / 7). Must be IDENTICAL for all 84 runs ----
EPOCHS        = 100               # step 12 kept 100 (pilot: still 0.10 mph better after epoch 50)
PATIENCE      = None              # no early stopping, as in step 12
RUNS_SUBDIR   = 'runs'            # 'runs' for 100 epochs ('runs_e50' would be for a 50-epoch project)
# ------------------------------------------------------------------------------------------------

# ---- this batch ----
SESSION_HOURS = None              # e.g. 3.5 = train for 3.5 h then stop cleanly; None = until Colab stops you
MAX_RUNS      = None              # e.g. 1 to finish one run and stop
ONLY_DATASET  = None              # 'PEMS-BAY', 'SHARJAH' or None (= both)
RECLAIM_MY_RUNS = True            # take back the run a crashed session of yours left 'in_progress' (section 6)

# ---- time planning (section 6) ----
GPU_HOURS_PER_DAY = 6             # hours one GPU really trains per day (free Colab 3-4, Pro with background execution up to 24)
N_GPUS            = 1             # sessions of this notebook running at the same time (each with its own WORKER)
DEADLINE          = '2026-11-21'  # the day the last run must be finished (so steps 16-17 and the report fit); CHANGE IT
COMPUTE_UNITS_LEFT = None         # Colab Pro: units left on the account (Resources panel), e.g. 420; None on a free account
UNITS_PER_HOUR     = None         # Colab Pro: what the current GPU costs per hour (Resources panel), e.g. 11.8 for an A100

# ---------------------------------------------------------------------------------
OWN_STALE_MIN = 10                # minutes without heartbeat after which a claim with MY name is treated as dead
WORKER = WORKER.strip().lower()
ORDER = +1 if (WORKER == 'mohammed' or ONLY_DATASET) else -1     # a second worker without ONLY_DATASET walks bottom-up
if (EPOCHS, PATIENCE) != (100, None) and RUNS_SUBDIR == 'runs':
    raise ValueError("EPOCHS/PATIENCE differ from the 100-epoch baseline: set RUNS_SUBDIR to its own folder, e.g. 'runs_e50', the same as step 12")
if (EPOCHS, PATIENCE) == (100, None) and RUNS_SUBDIR != 'runs':
    raise ValueError("100 epochs and no patience belong in RUNS_SUBDIR = 'runs' (where the step-3 baseline and the step-12 runs are)")
SETTINGS = dict(epochs=EPOCHS, patience=PATIENCE)
DEADLINE_DT = datetime.strptime(DEADLINE, '%Y-%m-%d').replace(hour=23, minute=59, tzinfo=TZ)
print(f'42 runs: {len(DATASETS)} datasets x 7 versions x 3 seeds, {EPOCHS} epochs'
      + (f', early stop after {PATIENCE} epochs without improvement' if PATIENCE else '')
      + f', in {RUNS_SUBDIR}/; worker {WORKER}' + (f', only {ONLY_DATASET}' if ONLY_DATASET else ''))
print(f'this batch: ' + (f'{SESSION_HOURS} h' if SESSION_HOURS else 'until no free run is left or Colab stops')
      + (f', at most {MAX_RUNS} run(s)' if MAX_RUNS is not None else ''))
print(f'planning: {N_GPUS} GPU(s) x {GPU_HOURS_PER_DAY} h/day, deadline {DEADLINE_DT:%a %d %b %Y}')

42 runs: 2 datasets x 7 versions x 3 seeds, 100 epochs, in runs/; worker mohammed
this batch: until no free run is left or Colab stops
planning: 1 GPU(s) x 6 h/day, deadline Sat 21 Nov 2026


## 3. Al Zahraa's training code (step 12), the scoring code (step 9) and Graph WaveNet

`code/train_gwnet.py` is written to Drive by section 3 of the step-12 notebook. This notebook **imports** it and never rewrites it, so both students run exactly the same training loop, resume logic and scoring hook. If the file is missing, ask Al Zahraa to run that section once (it only writes the file). `code/metrics.py` is step 9's scoring code, called by the module the moment a run finishes.

Then Graph WaveNet is fetched at the pinned commit with the same `Conv1d → Conv2d` patch as step 3 (the module does this), and the GPU is checked.

**Which GPU.** Take the fastest one Colab offers (A100 › L4 › T4). The training settings do not change with the GPU (batch 64 stays, the module fixes them), so a faster card only shortens the wall-clock time: on this model an A100 is roughly 3 to 5 times faster than a T4, an L4 about 1.5 to 2 times. The notebook uses such a factor only for the **first guess**; from the first epoch on, the ETA uses the time measured on the GPU you actually got (Colab sometimes hands out a lower one than you asked for; the name is printed below and recorded in every run's `results.json`, so the report's setup section can list it). Two things to know: a fast GPU costs several times more compute units per hour, so per **finished run** the T4 is usually the cheapest and the A100 the fastest; and numbers from different GPUs can differ in the last decimals (the A100 uses TF32 arithmetic in convolutions), well below the seed-to-seed noise.

In [ ]:
for f, fix in [('train_gwnet.py', 'ask Al Zahraa to run section 3 of Week3_Step12_train_seven_versions.ipynb once (it writes the file to Drive)'),
               ('metrics.py', 'run section 6 of Week2_Step9_evaluation.ipynb once')]:
    assert os.path.exists(os.path.join(CODE_DIR, f)), f'code/{f} is missing: {fix}'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import importlib, train_gwnet as T, metrics as M
T, M = importlib.reload(T), importlib.reload(M)
need = ['job_list', 'status_board', 'time_estimate', 'sec_per_epoch_estimate', 'work', 'graph_file', 'original_graph',
        'data_dir', 'run_dir', 'read_json', 'release_claim', 'setup_graph_wavenet', 'ORIGINAL_GRAPHS', 'GUESS_SEC_PER_EPOCH',
        'GRAPH_VERSIONS', 'SEEDS']
missing = [n for n in need if not hasattr(T, n)]
assert not missing, f'train_gwnet.py is not the step-12 version (missing {missing}): ask Al Zahraa to rerun step-12 section 3'
print('training code :', T.__file__, f'({os.path.getsize(T.__file__):,} bytes, saved {time.strftime("%d %b %H:%M", time.localtime(os.path.getmtime(T.__file__)))})')
print('scoring code  :', M.__file__)
print('fixed model settings (step 3):', {k: v for k, v in T.DEFAULTS.items() if k not in ('epochs', 'patience')})

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU found: Runtime -> Change runtime type -> GPU"
GWN_DIR = '/content/Graph-WaveNet' if IN_COLAB else os.path.join(os.path.expanduser('~'), 'Graph-WaveNet')
T.setup_graph_wavenet(GWN_DIR)
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
GPU_SPEED = {'B200': 7.0, 'H100': 5.0, 'A100': 3.5, 'A10': 2.0, 'L4': 1.7, 'V100': 1.6, 'P100': 1.2, 'T4': 1.0}   # rough speed vs a T4 on this model
GPU_FACTOR = next((f for k, f in GPU_SPEED.items() if k in GPU_NAME.upper()), 1.0)
print(f'GPU: {GPU_NAME} -> first-guess speed {GPU_FACTOR}x a T4 (the measured epoch time replaces this after the first epoch)')
if not torch.cuda.is_available():
    if os.environ.get('EXPRESS_ALLOW_CPU'):
        print('WARNING: no GPU, continuing on the CPU because EXPRESS_ALLOW_CPU is set (local test only)')
    else:
        raise RuntimeError('No GPU: Runtime -> Change runtime type -> GPU, then Run all.')

training code : /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding/code/train_gwnet.py (29,844 bytes, saved 07 Nov 19:17)
scoring code  : /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding/code/metrics.py
fixed model settings (step 3): {'batch_size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.3, 'nhid': 32, 'adj_type': 'doubletransition', 'adaptive_adj': False}
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
torch 2.14.1+cu130 | CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition -> first-guess speed 1.0x a T4 (the measured epoch time replaces this after the first epoch)


## 4. The two datasets: names, original graphs, size, and how long an epoch takes

Step 2 saved PEMS-BAY as `data/processed/PEMS-BAY/` with its original graph at `data/raw/adj_mx_bay.pkl`. Step 11 saved Sharjah as `data/processed/SHARJAH/` with its graph in its own raw folder, `data/raw/sharjah/SHARJAH_adj_mx.pkl`; the module's copy in Drive knows that path since 7 Nov, and if an older copy does not, this cell tells it (`T.ORIGINAL_GRAPHS['SHARJAH']`) instead of editing `train_gwnet.py`.

It also reads the number of sensors and samples of both datasets (from the graph pickle and the `t_index` arrays, so nothing big is loaded) and turns them into a **first guess of the seconds per epoch**: time per epoch is proportional to sensors × training samples, and the finished runs in `runs/results.csv` (METR-LA: 131 s for 207 sensors × 24k samples) give the constant. The guess is only used until the first real epoch of each dataset is timed; from then on section 6 uses the measured value.

In [ ]:
def resolve_name(name):
    """The dataset folder under data/processed/, matched without case, so 'SHARJAH' and 'Sharjah' both work."""
    root = os.path.join(PROJECT_DIR, 'data', 'processed')
    for d in (sorted(os.listdir(root)) if os.path.isdir(root) else []):
        if d.lower() == name.lower():
            return d
    return name

def load_ids(pkl):
    with open(pkl, 'rb') as fh:
        try:
            return pickle.load(fh)[0]
        except UnicodeDecodeError:
            fh.seek(0); return pickle.load(fh, encoding='latin1')[0]

def n_train_samples(ds):
    p = os.path.join(T.data_dir(PROJECT_DIR, ds), 'train.npz')
    return int(len(np.load(p)['t_index'])) if os.path.exists(p) else None

DATASETS = [resolve_name(d) for d in DATASETS]
ONLY_DATASET = resolve_name(ONLY_DATASET) if ONLY_DATASET else None
SHARJAH = next((d for d in DATASETS if d.lower().startswith('sharjah')), None)
assert SHARJAH, 'DATASETS must contain the Sharjah dataset'

# --- the Sharjah original graph (step 11 writes data/raw/sharjah/SHARJAH_adj_mx.pkl) ---
of = T.original_graph(PROJECT_DIR, SHARJAH)
if not os.path.exists(of):
    found = sorted(glob.glob(os.path.join(PROJECT_DIR, 'data', 'raw', '*', f'{SHARJAH}_adj_mx.pkl')))
    assert found, (f'The Sharjah original graph is missing: run Week3_Step11_sharjah.ipynb (step 11) first; '
                   f'it writes data/raw/sharjah/{SHARJAH}_adj_mx.pkl')
    T.ORIGINAL_GRAPHS[SHARJAH] = os.path.relpath(found[0], os.path.join(PROJECT_DIR, 'data', 'raw'))
    print(f'T.ORIGINAL_GRAPHS[{SHARJAH!r}] = {T.ORIGINAL_GRAPHS[SHARJAH]!r}   (this copy of the module looked elsewhere)')
    of = T.original_graph(PROJECT_DIR, SHARJAH)
assert os.path.exists(of)

# --- size of each dataset, from small files only ---
sizes = {}
for ds in DATASETS:
    og = T.original_graph(PROJECT_DIR, ds)
    assert os.path.exists(og), f'original graph missing for {ds}: {os.path.relpath(og, PROJECT_DIR)}'
    dd = T.data_dir(PROJECT_DIR, ds)
    n_samples = {p: int(len(np.load(os.path.join(dd, f'{p}.npz'))['t_index'])) for p in ['train', 'val', 'test']
                 if os.path.exists(os.path.join(dd, f'{p}.npz'))}
    sizes[ds] = dict(sensors=len(load_ids(og)), graph=os.path.relpath(og, PROJECT_DIR), **n_samples)
display(pd.DataFrame(sizes).T)

# --- seconds per epoch: measured runs of other datasets give "seconds per (sensor x training sample)" ---
K_FALLBACK = 131.0 / (207 * 23974)                     # the step-3 METR-LA run on a T4, if results.csv is not there
ref = []
res_csv = os.path.join(PROJECT_DIR, RUNS_SUBDIR, 'results.csv')
if os.path.exists(res_csv):
    r = pd.read_csv(res_csv)
    if 'gpu' not in r:
        r['gpu'] = 'unknown'
    for (ds, gpu), g in r.groupby(['dataset', r['gpu'].fillna('unknown')]):
        og = T.original_graph(PROJECT_DIR, ds)
        n_tr = n_train_samples(ds)
        if os.path.exists(og) and n_tr and g['sec_per_epoch'].notna().any():
            spe = float(g['sec_per_epoch'].median()); n_s = len(load_ids(og))
            ref.append(dict(dataset=ds, gpu=gpu, runs=len(g), sec_per_epoch=round(spe, 1), sensors=n_s, train_samples=n_tr,
                            sec_per_sensor_sample=spe / (n_s * n_tr)))
same_gpu = [x for x in ref if x['gpu'] == GPU_NAME]
if same_gpu:
    K, how = float(np.median([x['sec_per_sensor_sample'] for x in same_gpu])), f'from finished runs on this GPU ({GPU_NAME})'
elif ref:
    K = float(np.median([x['sec_per_sensor_sample'] for x in ref])) / GPU_FACTOR
    how = f'from finished runs on {sorted(set(x["gpu"] for x in ref))}, divided by {GPU_FACTOR} for the {GPU_NAME}'
else:
    K, how = K_FALLBACK / GPU_FACTOR, f'T4 fallback (METR-LA 131 s) divided by {GPU_FACTOR} for the {GPU_NAME}'
if ref:
    display(pd.DataFrame(ref).set_index(['dataset', 'gpu']))
print(f'seconds per epoch = {K:.3e} x sensors x training samples   ({how})')
guess = []
for ds in DATASETS:
    if sizes[ds].get('train'):
        T.GUESS_SEC_PER_EPOCH[ds] = int(round(K * sizes[ds]['sensors'] * sizes[ds]['train']))
    spe = T.GUESS_SEC_PER_EPOCH.get(ds, T.GUESS_SEC_PER_EPOCH['TOY'])
    guess.append(dict(dataset=ds, sec_per_epoch=spe, min_per_epoch=round(spe / 60, 1),
                      hours_per_run=round((EPOCHS * spe + 4 * 60) / 3600, 1), hours_21_runs=round(21 * (EPOCHS * spe + 240) / 3600)))
display(pd.DataFrame(guess).set_index('dataset'))
print(f'first guess for the 42 runs at {EPOCHS} epochs on the {GPU_NAME}: about {sum(g["hours_21_runs"] for g in guess)} GPU-hours; '
      'replaced by measured times as soon as an epoch of each dataset has run (section 6)')

,sensors,graph,train,val,test
PEMS-BAY,325,data/raw/adj_mx_bay.pkl,36465,5209,10419
SHARJAH,300,data/raw/sharjah/SHARJAH_adj_mx.pkl,24176,3454,6907


,,runs,sec_per_epoch,sensors,train_samples,sec_per_sensor_sample
dataset,gpu,,,,,
METR-LA,Tesla T4,1,131.0,207,23974,0.000026
PEMS-BAY,NVIDIA RTX PRO 6000 Blackwell Server Edition,2,27.0,325,36465,0.000002
TOY-C1,Tesla T4,2,130.3,200,24176,0.000027


seconds per epoch = 2.278e-06 x sensors x training samples   (from finished runs on this GPU (NVIDIA RTX PRO 6000 Blackwell Server Edition))


,sec_per_epoch,min_per_epoch,hours_per_run,hours_21_runs
dataset,,,,
PEMS-BAY,27,0.5,0.8,17
SHARJAH,17,0.3,0.5,11


first guess for the 42 runs at 100 epochs on the NVIDIA RTX PRO 6000 Blackwell Server Edition: about 28 GPU-hours; replaced by measured times as soon as an epoch of each dataset has run (section 6)


## 5. Check the inputs and the job list

Graph versions are expected at `data/graphs/<dataset>/<version>_K25.pkl` with `<version>` in `random, map_only, corr_only, same_side, express, virtual_node` (`original` is the raw road graph). The sensor order must match the original graph; the `virtual_node` file may have one extra node at the end, which the module pads with an always-masked data column. **Missing files are Al Zahraa's step 10**; they show as `blocked` below and are skipped, not failed, so run what exists and rerun the notebook when the rest arrive.

The scoring code finds the behind-a-bottleneck subset in one of two places: `data/processed/<dataset>/bottleneck/regions.npz` (step 5, used for PEMS-BAY) or `data/raw/toy/<dataset>_jam_arrivals.csv` (the per-jam `crossed` flag of step 8). Step 11 saved Sharjah's jam log in `data/raw/sharjah/`, so this cell copies that one file to where `metrics.py` looks; nothing in `metrics.py` changes. If a subset file is missing, the runs still train and get their error metrics (MAE, RMSE, MAPE: mean absolute error, root mean square error, mean absolute percentage error); only the `*_behind` columns stay empty until the file appears and the run is rescored (section 8 rescores everything).

Finally, the step-12 runs already in Drive are read to make sure `EPOCHS` and `PATIENCE` here are the same as there.

In [ ]:
JOBS = T.job_list(DATASETS)
print(f'{len(JOBS)} jobs: {len(DATASETS)} datasets x {len(T.GRAPH_VERSIONS)} versions x {len(T.SEEDS)} seeds')

rows = []
for ds in DATASETS:
    dd = T.data_dir(PROJECT_DIR, ds)
    ok = all(os.path.exists(os.path.join(dd, f'{s}.npz')) for s in ['train', 'val', 'test'])
    rows.append(dict(dataset=ds, what='data (step 2 / step 11)', file=os.path.relpath(dd, PROJECT_DIR), exists=ok))
    for v in T.GRAPH_VERSIONS:
        gf = T.graph_file(PROJECT_DIR, ds, v)
        rows.append(dict(dataset=ds, what=('graph: original' if v == 'original' else f'graph: {v} (step 10)'),
                         file=os.path.relpath(gf, PROJECT_DIR), exists=os.path.exists(gf)))

# --- the behind-a-bottleneck files that metrics.score_run looks for ---
RAW_TOY = os.path.join(PROJECT_DIR, 'data', 'raw', 'toy')
for ds in DATASETS:
    dd = T.data_dir(PROJECT_DIR, ds)
    reg = os.path.join(dd, 'bottleneck', 'regions.npz')
    arr = os.path.join(RAW_TOY, f'{ds}_jam_arrivals.csv')
    if ds == SHARJAH:
        src = sorted(glob.glob(os.path.join(PROJECT_DIR, 'data', 'raw', '*', f'{ds}_jam_arrivals.csv')))
        src = [s for s in src if os.path.abspath(s) != os.path.abspath(arr)]
        if src and (not os.path.exists(arr) or not filecmp.cmp(src[0], arr, shallow=False)):
            os.makedirs(RAW_TOY, exist_ok=True)
            shutil.copyfile(src[0], arr)
            print(f'copied {os.path.relpath(src[0], PROJECT_DIR)} -> {os.path.relpath(arr, PROJECT_DIR)} (where metrics.py looks)')
        ok = os.path.exists(arr)
        if ok:
            cols = set(pd.read_csv(arr, nrows=1).columns)
            assert {'road', 'arrival_row', 'crossed'} <= cols, f'{arr} lacks the columns metrics.py needs (road, arrival_row, crossed)'
            n_cross = int(pd.read_csv(arr)['crossed'].sum())
            print(f'{ds}: jam log with {n_cross} crossings of the lagoon; subset = jams that crossed (step 11)')
        rows.append(dict(dataset=ds, what='behind-bottleneck: jam log (step 11)', file=os.path.relpath(arr, PROJECT_DIR), exists=ok))
    else:
        ok = os.path.exists(reg)
        if ok:
            r = np.load(reg, allow_pickle=True)
            assert 'behind_bottleneck' in r, f'{reg} has no behind_bottleneck array'
            print(f'{ds}: {int(r["behind_bottleneck"].sum())} of {len(r["behind_bottleneck"])} roads behind the bottleneck (step 5)')
        rows.append(dict(dataset=ds, what='behind-bottleneck: regions.npz (step 5)', file=os.path.relpath(reg, PROJECT_DIR), exists=ok))
rows.append(dict(dataset='-', what='scoring (step 9)', file='code/metrics.py', exists=os.path.exists(os.path.join(CODE_DIR, 'metrics.py'))))
rows.append(dict(dataset='-', what='training (step 12)', file='code/train_gwnet.py', exists=os.path.exists(os.path.join(CODE_DIR, 'train_gwnet.py'))))
inputs = pd.DataFrame(rows)
display(inputs)
missing = inputs[~inputs['exists']]
for r in missing.itertuples():
    if 'step 10' in r.what:
        print(f'MISSING {r.file}: step 10 (Al Zahraa); the {r.dataset} {r.what.split()[1]} runs show as blocked until it appears')
    elif 'step 5' in r.what:
        print(f'MISSING {r.file}: step 5 (Mohammed); {r.dataset} trains and gets its MAE, but the *_behind columns stay empty until it exists')
    elif 'step 11' in r.what:
        print(f'MISSING {r.file}: step 11 (Mohammed); {r.dataset} trains, but the *_behind columns stay empty until the jam log exists')
    else:
        print(f'MISSING {r.file} ({r.what}): training cannot start for {r.dataset}')
if not len(missing):
    print('OK  every input is in place')

# --- same training length as the step-12 runs already in Drive? ---
others = []
for p in glob.glob(os.path.join(PROJECT_DIR, RUNS_SUBDIR, '*', '*', 'seed*', 'results.json')):
    r = T.read_json(p, {})
    if r.get('step12') and r.get('dataset') not in DATASETS and 'epochs' in r:
        others.append((r['dataset'], r['graph'], r['seed'], r['epochs'], r.get('patience')))
if others:
    diff = [o for o in others if (o[3], o[4]) != (EPOCHS, PATIENCE)]
    assert not diff, (f'{len(diff)} step-12 run(s) in {RUNS_SUBDIR}/ used epochs={diff[0][3]}, patience={diff[0][4]} '
                      f'(e.g. {diff[0][0]}/{diff[0][1]}/seed{diff[0][2]}); set EPOCHS / PATIENCE / RUNS_SUBDIR to the same values')
    print(f'OK  EPOCHS={EPOCHS}, PATIENCE={PATIENCE} match the {len(others)} finished step-12 run(s) in {RUNS_SUBDIR}/')
else:
    print(f'No finished step-12 run in {RUNS_SUBDIR}/ yet: check EPOCHS / PATIENCE / RUNS_SUBDIR with Al Zahraa before training')

42 jobs: 2 datasets x 7 versions x 3 seeds
PEMS-BAY: 0 of 325 roads behind the bottleneck (step 5)
SHARJAH: jam log with 5275 crossings of the lagoon; subset = jams that crossed (step 11)


,dataset,what,file,exists
0,PEMS-BAY,data (step 2 / step 11),data/processed/PEMS-BAY,True
1,PEMS-BAY,graph: original,data/raw/adj_mx_bay.pkl,True
2,PEMS-BAY,graph: random (step 10),data/graphs/PEMS-BAY/random_K25.pkl,True
3,PEMS-BAY,graph: map_only (step 10),data/graphs/PEMS-BAY/map_only_K25.pkl,True
4,PEMS-BAY,graph: corr_only (step 10),data/graphs/PEMS-BAY/corr_only_K25.pkl,True
5,PEMS-BAY,graph: same_side (step 10),data/graphs/PEMS-BAY/same_side_K25.pkl,True
6,PEMS-BAY,graph: express (step 10),data/graphs/PEMS-BAY/express_K25.pkl,True
7,PEMS-BAY,graph: virtual_node (step 10),data/graphs/PEMS-BAY/virtual_node_K25.pkl,True
8,SHARJAH,data (step 2 / step 11),data/processed/SHARJAH,True
9,SHARJAH,graph: original,data/raw/sharjah/SHARJAH_adj_mx.pkl,True


OK  every input is in place
OK  EPOCHS=100, PATIENCE=None match the 2 finished step-12 run(s) in runs/


## 6. Where we are, the ETA of every run, and does it fit before the deadline?

Statuses: **done** (forecasts + results written), **test_pending** (all epochs trained, only the test pass left: minutes), **in_progress** (another session holds a live claim), **resumable** (a checkpoint and no live claim: the next worker continues it), **not_started**, **blocked** (graph file missing).

The ETA table lists the runs **in the order this worker will take them** (seed 0 of every version first; blocked runs last, after their graph file arrives). For each run: epochs done, hours left, the GPU-hours until it is finished, and the expected finish date at `GPU_HOURS_PER_DAY` hours a day on `N_GPUS` GPUs. The seconds per epoch are the measured median of any run of the same dataset in Drive, or the section-4 guess until the first epoch is timed (the `source` column says which).

**If it does not fit before `DEADLINE`**, decide together and write the decision in the report. The plan's order of cuts: (1) seed 0 of all 14 runs first (the job order already does this, so whatever finishes is a complete block); (2) more GPUs at once (a second Colab account, the GPU server overnight), each with its own `WORKER`; (3) only then fewer epochs, for **all** 84 runs of the project, in a separate `runs_e50/` folder, with a note in the report. Never drop the same-side control, and never mix epoch counts between versions or datasets.

Everything printed here is also saved to `runs/progress_step13.txt` and `runs/progress_step13.csv` in Drive.

In [ ]:
TEST_MIN = 4.0                                   # minutes for the test pass of one run (as in T.time_estimate)

def hours_left_of(status, epoch, epochs, spe):
    if status == 'done':
        return 0.0
    return (max(0, epochs - min(int(epoch), epochs)) * spe + TEST_MIN * 60) / 3600

SESSION_SPE = globals().get('SESSION_SPE', {})       # seconds per epoch measured in THIS session, per dataset (section 7 fills it)

def spe_for(board, ds):
    """Seconds per epoch for a dataset: measured on this GPU in this session > measured in earlier runs (any GPU) > the section-4 guess."""
    if SESSION_SPE.get(ds):
        return float(np.median(SESSION_SPE[ds][-10:])), 'measured on this GPU'
    spe, how = T.sec_per_epoch_estimate(board, ds, EPOCHS)
    return spe, ('measured (earlier runs)' if how == 'measured' else f'guess ({GPU_NAME})')

def eta_table(board, epochs=None, seeds=None):
    """One row per run in this worker's walking order: hours left, GPU-hours until finished, expected finish date."""
    epochs = epochs or EPOCHS
    b = board.copy()
    if seeds is not None:
        b = b[b['seed'].isin(seeds)]
    spe = {ds: spe_for(board, ds) for ds in b['dataset'].unique()}
    b['sec_per_epoch_used'] = [round(spe[d][0]) for d in b['dataset']]
    b['source'] = [spe[d][1] for d in b['dataset']]
    b['hours_left'] = [hours_left_of(s, e, epochs, spe[d][0]) for s, e, d in zip(b['status'], b['epoch'], b['dataset'])]
    seq = b if ORDER > 0 else b.iloc[::-1]
    if ONLY_DATASET:                                     # my dataset first, the other one after it
        seq = pd.concat([seq[seq['dataset'] == ONLY_DATASET], seq[seq['dataset'] != ONLY_DATASET]])
    seq = pd.concat([seq[seq['status'] != 'blocked'], seq[seq['status'] == 'blocked']])
    seq = seq.reset_index(drop=True)
    seq['gpu_hours_until_done'] = seq['hours_left'].cumsum()
    per_day = max(GPU_HOURS_PER_DAY * N_GPUS, 1e-9)
    t0 = now_local()
    seq['expected_finish'] = ['done' if s == 'done' else (t0 + timedelta(days=h / per_day)).strftime('%a %d %b')
                              for s, h in zip(seq['status'], seq['gpu_hours_until_done'])]
    seq.loc[seq['status'] == 'blocked', 'expected_finish'] = 'after the step-10 file'
    return seq

def fmt_h(h):
    return f'{h:.1f} h' if h < 10 else f'{h:.0f} h'

def eta_report(board):
    """The ETA table and a few summary lines (printed in section 6, after every run in section 7, saved to Drive)."""
    eta = eta_table(board)
    counts = eta['status'].value_counts()
    left = eta[eta['status'] != 'done']
    total_h = float(left['hours_left'].sum())
    per_day = GPU_HOURS_PER_DAY * N_GPUS
    days_left = max(0.0, (DEADLINE_DT - now_local()).total_seconds() / 86400)
    budget_h = days_left * per_day
    by_ds = ' + '.join(f'{ds} {fmt_h(g["hours_left"].sum())} ({len(g)} runs)' for ds, g in left.groupby('dataset', sort=False))
    takeable = left[left['status'] != 'blocked']
    fits_n = int((takeable['gpu_hours_until_done'] <= budget_h).sum())
    guessed = sorted(set(eta.loc[eta['source'].str.startswith('guess'), 'dataset']))
    lines = [
        f'{now_local():%a %d %b %H:%M} | {counts.get("done", 0)}/{len(eta)} runs done'
        + ''.join(f', {counts[s]} {s}' for s in ['in_progress', 'resumable', 'test_pending', 'not_started', 'blocked'] if s in counts)
        + f' | GPU time left: {fmt_h(total_h)} ({by_ds})' + (f'  [{", ".join(guessed)}: guess until its first epoch is timed]' if guessed else ''),
        f'= {total_h / 24:.1f} days of one GPU running 24 h/day; at {N_GPUS} GPU(s) x {GPU_HOURS_PER_DAY} h/day: {total_h / per_day:.0f} days -> '
        f'all done about {(now_local() + timedelta(days=total_h / per_day)):%a %d %b %Y}',
        f'deadline {DEADLINE_DT:%a %d %b}: {days_left:.1f} days x {per_day} GPU-h/day = {fmt_h(budget_h)} available -> '
        + (f'FITS ({fmt_h(total_h)} needed)' if total_h <= budget_h else
           f'DOES NOT FIT ({fmt_h(total_h)} needed): about {fits_n} of the {len(takeable)} runs left will be finished by then'),
    ]
    if COMPUTE_UNITS_LEFT and UNITS_PER_HOUR:
        afford = COMPUTE_UNITS_LEFT / UNITS_PER_HOUR
        afford_n = int((takeable['gpu_hours_until_done'] <= afford).sum())
        e0 = eta_table(board, seeds=[T.SEEDS[0]])
        seed0_h = float(e0.loc[e0['status'] != 'done', 'hours_left'].sum())
        lines.append(f'compute units: {COMPUTE_UNITS_LEFT:g} left at {UNITS_PER_HOUR:g} per hour on the {GPU_NAME} = {fmt_h(afford)} of GPU -> '
                     + (f'COVERS the {fmt_h(total_h)} needed' if afford >= total_h else
                        f'covers about {afford_n} of the {len(takeable)} runs left ({fmt_h(total_h)} needed; seed 0 alone needs {fmt_h(seed0_h)})'))
    return eta, total_h, lines

def save_progress(eta, lines):
    try:
        out = os.path.join(PROJECT_DIR, RUNS_SUBDIR)
        cols = ['dataset', 'graph', 'seed', 'status', 'epoch', 'hours_left', 'gpu_hours_until_done', 'expected_finish',
                'sec_per_epoch_used', 'source', 'worker', 'scored']
        e = eta[cols].copy(); e.insert(0, 'written', now_local().strftime('%Y-%m-%d %H:%M'))
        e[['hours_left', 'gpu_hours_until_done']] = e[['hours_left', 'gpu_hours_until_done']].round(2)
        e.to_csv(os.path.join(out, 'progress_step13.csv'), index=False)
        with open(os.path.join(out, 'progress_step13.txt'), 'w') as f:
            f.write('\n'.join(lines) + '\n\n' + e.drop(columns=['written']).to_string(index=False) + '\n')
    except Exception as ex:
        print(f'(progress file not written: {ex})')

# --- take back the run a crashed session of mine left 'in_progress' ---
if RECLAIM_MY_RUNS:
    for j in JOBS:
        rd = T.run_dir(PROJECT_DIR, j['dataset'], j['graph'], j['seed'], RUNS_SUBDIR)
        c = T.read_json(os.path.join(rd, 'claim.json'), {})
        if not c or c.get('worker') != WORKER or T.read_json(os.path.join(rd, 'results.json'), {}).get('step12'):
            continue
        name = f'{j["dataset"]}/{j["graph"]}/seed{j["seed"]}'
        age = time.time() - c.get('heartbeat', 0)
        if age < OWN_STALE_MIN * 60:
            wait = OWN_STALE_MIN * 60 - age
            print(f'{name}: a session of yours wrote a heartbeat {age / 60:.0f} min ago; waiting {wait / 60:.0f} min to be sure it is dead...')
            time.sleep(wait)
            c2 = T.read_json(os.path.join(rd, 'claim.json'), {})
            if not c2:
                continue                                  # it finished or stopped cleanly meanwhile
            if c2.get('heartbeat') != c.get('heartbeat'):
                print(f'{name}: new heartbeat, that session is alive; leaving it alone (give each session its own WORKER name)')
                continue
        T.release_claim(rd)
        print(f'took back {name} (your last heartbeat {c.get("time")}); it resumes in its turn below')

# --- the board ---
board = T.status_board(PROJECT_DIR, JOBS, RUNS_SUBDIR, EPOCHS)
eta, total_h, lines = eta_report(board)
print('\n'.join(lines))
save_progress(eta, lines)

show = eta[['dataset', 'graph', 'seed', 'status', 'epoch', 'hours_left', 'gpu_hours_until_done', 'expected_finish',
            'sec_per_epoch_used', 'source', 'worker', 'scored']].copy()
show['epoch'] = [f'{e}/{EPOCHS}' for e in show['epoch']]
show[['hours_left', 'gpu_hours_until_done']] = show[['hours_left', 'gpu_hours_until_done']].round(1)
display(show.style.apply(lambda s: ['background-color: #d9f2e3' if v == 'done' else
                                    'background-color: #fde9d0' if v == 'in_progress' else
                                    'background-color: #e8e8e8' if v == 'blocked' else '' for v in s], subset=['status']))

# --- the plan's cut options, priced the same way ---
def hours_for(epochs, seeds):
    e = eta_table(board, epochs=epochs, seeds=seeds)
    return float(e.loc[e['status'] != 'done', 'hours_left'].sum())
days_left = max(0.0, (DEADLINE_DT - now_local()).total_seconds() / 86400)
budget_h = days_left * GPU_HOURS_PER_DAY * N_GPUS
options = [(f'all {len(JOBS)} runs at {EPOCHS} epochs (the plan)', hours_for(EPOCHS, T.SEEDS)),
           (f'seed 0 only ({len(JOBS) // len(T.SEEDS)} runs) at {EPOCHS} epochs (cut 1)', hours_for(EPOCHS, [T.SEEDS[0]]))]
if EPOCHS > 50:
    options += [(f'all {len(JOBS)} runs at 50 epochs (cut 3: then ALL 84 runs must be 50)', hours_for(50, T.SEEDS)),
                ('seed 0 only at 50 epochs', hours_for(50, [T.SEEDS[0]]))]
cuts = pd.DataFrame([dict(option=o, gpu_hours=round(h), days_at_your_rate=round(h / (GPU_HOURS_PER_DAY * N_GPUS), 1),
                          fits_before_deadline=h <= budget_h) for o, h in options])
display(cuts)
if total_h > budget_h:
    print(f'Cut order from the plan: (1) seed 0 of all 14 first (already the job order); (2) more GPUs at once (N_GPUS), each with its own WORKER;')
    print('(3) only then fewer epochs for ALL runs of the project (same change in step 12, RUNS_SUBDIR = "runs_e50", note in the report).')
    print('Never drop the same-side control; never mix epoch counts between versions or datasets.')

# --- what this session will do ---
if SESSION_HOURS:
    budget, plan = SESSION_HOURS, []
    for r in eta[(eta['status'] != 'done') & (eta['status'] != 'blocked') & (eta['status'] != 'in_progress')].itertuples():
        name = f'{r.dataset}/{r.graph}/seed{r.seed}'
        if r.hours_left <= budget:
            plan.append(f'{name}: finishes ({r.hours_left:.1f} h' + (f', resuming at epoch {r.epoch + 1}' if r.epoch else '') + ')')
            budget -= r.hours_left
        else:
            reach = int(r.epoch) + int(budget * 3600 // r.sec_per_epoch_used)
            plan.append(f'{name}: reaches about epoch {min(reach, EPOCHS)}/{EPOCHS}, then stops cleanly'); break
    print(f'\nThis session ({SESSION_HOURS} h) should: ' + ('\n  ' + '\n  '.join(plan) if plan else 'nothing left to take'))
else:
    nxt = eta[(eta['status'] != 'done') & (eta['status'] != 'blocked') & (eta['status'] != 'in_progress')]
    if len(nxt):
        r = nxt.iloc[0]
        print(f'\nNext run: {r["dataset"]}/{r["graph"]}/seed{r["seed"]} ({r["status"]}, {r["hours_left"]:.1f} h left); '
              'the session continues until no free run is left or Colab stops it (set SESSION_HOURS to stop cleanly before that).')

Sun 08 Nov 19:02 | 2/42 runs done, 1 resumable, 39 not_started | GPU time left: 24 h (PEMS-BAY 13 h (19 runs) + SHARJAH 11 h (21 runs))  [SHARJAH: guess until its first epoch is timed]
= 1.0 days of one GPU running 24 h/day; at 1 GPU(s) x 6 h/day: 4 days -> all done about Thu 12 Nov 2026
deadline Sat 21 Nov: 13.2 days x 6 GPU-h/day = 79 h available -> FITS (24 h needed)


,dataset,graph,seed,status,epoch,hours_left,gpu_hours_until_done,expected_finish,sec_per_epoch_used,source,worker,scored
0,PEMS-BAY,original,0,done,100/100,0.000000,0.000000,done,23,measured on this GPU,,True
1,PEMS-BAY,random,0,done,100/100,0.000000,0.000000,done,23,measured on this GPU,,True
2,PEMS-BAY,map_only,0,resumable,69/100,0.300000,0.300000,Sun 08 Nov,23,measured on this GPU,,False
3,PEMS-BAY,corr_only,0,not_started,0/100,0.700000,1.000000,Sun 08 Nov,23,measured on this GPU,,False
4,PEMS-BAY,same_side,0,not_started,0/100,0.700000,1.700000,Mon 09 Nov,23,measured on this GPU,,False
5,PEMS-BAY,express,0,not_started,0/100,0.700000,2.400000,Mon 09 Nov,23,measured on this GPU,,False
6,PEMS-BAY,virtual_node,0,not_started,0/100,0.700000,3.100000,Mon 09 Nov,23,measured on this GPU,,False
7,SHARJAH,original,0,not_started,0/100,0.500000,3.600000,Mon 09 Nov,17,guess (NVIDIA RTX PRO 6000 Blackwell Server Edition),,False
8,SHARJAH,random,0,not_started,0/100,0.500000,4.200000,Mon 09 Nov,17,guess (NVIDIA RTX PRO 6000 Blackwell Server Edition),,False
9,SHARJAH,map_only,0,not_started,0/100,0.500000,4.700000,Mon 09 Nov,17,guess (NVIDIA RTX PRO 6000 Blackwell Server Edition),,False


,option,gpu_hours,days_at_your_rate,fits_before_deadline
0,all 42 runs at 100 epochs (the plan),24,4.0,True
1,seed 0 only (14 runs) at 100 epochs (cut 1),7,1.1,True
2,all 42 runs at 50 epochs (cut 3: then ALL 84 r...,13,2.2,True
3,seed 0 only at 50 epochs,4,0.6,True



Next run: PEMS-BAY/map_only/seed0 (resumable, 0.3 h left); the session continues until no free run is left or Colab stops it (set SESSION_HOURS to stop cleanly before that).


## 7. Train (one batch)

Takes the first run that is not done, trains or resumes it, tests the best epoch, writes the results row, **scores it with `metrics.score_run` the moment it finishes**, releases the claim, and takes the next one. Stops when there is no free run left, at `SESSION_HOURS` (cleanly, after the current epoch) or at `MAX_RUNS`.

Each epoch line ends with the time left in that run; after every finished run, and when the session stops, one line gives the time left overall (from the epoch times measured on this GPU in this session) and the progress file in Drive is updated. If the notebook is interrupted, run it again: it continues after the last finished epoch.

In [ ]:
EPOCH_LINE = re.compile(r'^([^/\s]+)/[^/\s]+/seed\d+ epoch +\d+/\d+ \|.*\| (\d+) s \| ~')   # the module's per-epoch line

def log(msg):
    print(msg)
    m = EPOCH_LINE.match(msg)
    if m:                                                      # remember this GPU's real seconds per epoch for the ETA
        SESSION_SPE.setdefault(m.group(1), []).append(float(m.group(2)))
    if msg.startswith('\n===') or msg.startswith('Session time budget') or msg.startswith('All runs are done') or msg.startswith('No free run'):
        try:
            b = T.status_board(PROJECT_DIR, JOBS, RUNS_SUBDIR, EPOCHS)
            if msg.startswith('Session time budget'):           # this session's own claim is released a moment later
                b.loc[(b['worker'] == WORKER) & (b['status'] == 'in_progress'), 'status'] = 'resumable'
            e, h, lines = eta_report(b)
            save_progress(e, lines)
            print('    overall: ' + lines[0].split(' | ', 1)[1])
            print('    ' + lines[1])
        except Exception as ex:
            print(f'    (overall progress skipped: {ex})')

t_session = time.time()
n_done = T.work(PROJECT_DIR, JOBS, WORKER, order=ORDER, settings=SETTINGS, runs_subdir=RUNS_SUBDIR,
                max_runs=MAX_RUNS, session_hours=SESSION_HOURS, only_dataset=ONLY_DATASET, log=log)
print(f'\n{WORKER} finished {n_done} run(s) in this session ({(time.time() - t_session) / 3600:.1f} h). '
      'Everything is saved in Drive; Run all again to continue.')


=== mohammed takes PEMS-BAY/map_only/seed0 (resumable) ===
    overall: 2/42 runs done, 1 in_progress, 39 not_started | GPU time left: 24 h (PEMS-BAY 13 h (19 runs) + SHARJAH 11 h (21 runs))  [SHARJAH: guess until its first epoch is timed]
    = 1.0 days of one GPU running 24 h/day; at 1 GPU(s) x 6 h/day: 4 days -> all done about Thu 12 Nov 2026
Resuming PEMS-BAY/map_only/seed0 after epoch 69 (best val MAE so far 1.6466)
PEMS-BAY/map_only/seed0 epoch  70/100 | train MAE 1.521 | val MAE 1.686 RMSE 3.544 MAPE 3.84% | 23 s | ~0.2 h left in this run
PEMS-BAY/map_only/seed0 epoch  71/100 | train MAE 1.522 | val MAE 1.653 RMSE 3.485 MAPE 3.78% | 23 s | ~0.2 h left in this run
PEMS-BAY/map_only/seed0 epoch  72/100 | train MAE 1.519 | val MAE 1.736 RMSE 3.640 MAPE 4.34% | 23 s | ~0.2 h left in this run
PEMS-BAY/map_only/seed0 epoch  73/100 | train MAE 1.519 | val MAE 1.652 RMSE 3.445 MAPE 3.89% | 23 s | ~0.2 h left in this run
PEMS-BAY/map_only/seed0 epoch  74/100 | train MAE 1.516 | val MAE 

## 8. The full picture: all four datasets, 84 runs (Results)

The board over step 12's jobs (toy city, METR-LA) and this notebook's (PEMS-BAY, Sharjah), then step 9's `score_many` over every run folder that has forecasts. It rebuilds `runs/scores.csv` (one row per run) and `runs/scores_summary.csv` (mean (std) over seeds per dataset × version), and shows the one table the project is about: **jam recall at 60 minutes behind the bottleneck, by dataset and graph version**. Rerun this section whenever runs finish; rows are replaced, not duplicated.

In [ ]:
STEP12_DATASETS = ['TOY-C1', 'METR-LA']                    # Al Zahraa's two (step 12)
ALL_DATASETS = [resolve_name(d) for d in STEP12_DATASETS] + DATASETS
full = T.status_board(PROJECT_DIR, T.job_list(ALL_DATASETS), RUNS_SUBDIR, EPOCHS)
display(pd.crosstab(full['dataset'], full['status']).reindex(ALL_DATASETS))
print(f'{(full["status"] == "done").sum()} of {len(full)} runs done, {int(full["scored"].sum())} scored')
est_all, left_all, _ = T.time_estimate(full, EPOCHS, workers=2)
display(est_all)
print(f'GPU time left over all four datasets: about {left_all:.0f} h = {left_all / 24:.1f} days of one GPU day and night; '
      f'with two people training: {left_all / 2:.0f} h each')

status,done,not_started,resumable
dataset,,,
TOY-C1,2,17,2
METR-LA,21,0,0
PEMS-BAY,21,0,0
SHARJAH,21,0,0


65 of 84 runs done, 65 scored


,dataset,runs_left,epochs_left,sec_per_epoch,source,gpu_hours_left
0,TOY-C1,19,1843,134,measured,69.7
1,METR-LA,0,0,47,measured,0.0
2,PEMS-BAY,0,0,23,measured,0.0
3,SHARJAH,0,0,13,measured,0.0


GPU time left over all four datasets: about 70 h = 2.9 days of one GPU day and night; with two people training: 35 h each


In [ ]:
# Rebuild runs/scores.csv and the mean (std) table from every run with forecasts, on all four datasets (step-9 code)
run_dirs = sorted(os.path.dirname(p) for p in glob.glob(os.path.join(PROJECT_DIR, RUNS_SUBDIR, '*', '*', 'seed*', 'test_predictions.npz'))
                  if '/check-' not in p.replace(os.sep, '/'))
print(f'{len(run_dirs)} run folder(s) with forecasts')
if run_dirs:
    table = M.score_many(run_dirs, os.path.join(PROJECT_DIR, 'data', 'processed'), RAW_TOY,
                         table_path=os.path.join(PROJECT_DIR, RUNS_SUBDIR, 'scores.csv'))
    cols = [c for c in ['dataset', 'graph', 'seed', 'mae_15', 'mae_30', 'mae_60', 'jam_recall_30', 'jam_recall_60',
                        'jam_precision_60', 'jam_recall_60_behind', 'mae_60_behind', 'subset_source'] if c in table.columns]
    display(table[cols].round(3))
    summary = M.summarise(table)
    summary.to_csv(os.path.join(PROJECT_DIR, RUNS_SUBDIR, 'scores_summary.csv'))
    display(summary)
    if 'jam_recall_60_behind' in summary.columns:
        print('Jam recall at 60 min, roads / jams behind the bottleneck: mean (std) over seeds')
        display(summary['jam_recall_60_behind'].unstack('graph').reindex(columns=[g for g in T.GRAPH_VERSIONS
                                                                                 if g in summary.index.get_level_values('graph')]))
else:
    print('No forecasts yet: section 7 writes runs/<dataset>/<graph>/seed<k>/test_predictions.npz for every finished run.')

65 run folder(s) with forecasts


,dataset,graph,seed,mae_15,mae_30,mae_60,jam_recall_30,jam_recall_60,jam_precision_60,jam_recall_60_behind,mae_60_behind,subset_source
0,METR-LA,corr_only,0,2.712,3.110,3.588,0.199,0.115,0.586,0.0,2.483,step 5 regions.npz (behind_bottleneck)
1,METR-LA,corr_only,1,2.729,3.128,3.588,0.178,0.082,0.608,0.0,2.459,step 5 regions.npz (behind_bottleneck)
2,METR-LA,corr_only,2,2.719,3.113,3.584,0.199,0.075,0.554,0.0,2.446,step 5 regions.npz (behind_bottleneck)
3,METR-LA,express,0,2.713,3.099,3.565,0.204,0.097,0.564,0.0,2.449,step 5 regions.npz (behind_bottleneck)
4,METR-LA,express,1,2.718,3.094,3.542,0.205,0.083,0.596,0.0,2.452,step 5 regions.npz (behind_bottleneck)
...,...,...,...,...,...,...,...,...,...,...,...,...
60,SHARJAH,virtual_node,0,0.907,1.113,1.463,0.337,0.000,NaN,0.0,NaN,step 8 SHARJAH_jam_arrivals.csv (crossed; 100%...
61,SHARJAH,virtual_node,1,0.908,1.116,1.471,0.408,0.000,NaN,0.0,NaN,step 8 SHARJAH_jam_arrivals.csv (crossed; 100%...
62,SHARJAH,virtual_node,2,0.909,1.113,1.453,0.365,0.000,NaN,0.0,NaN,step 8 SHARJAH_jam_arrivals.csv (crossed; 100%...
63,TOY-C1,original,0,0.889,1.108,1.539,0.555,0.017,0.800,0.0,NaN,step 8 TOY-C1_jam_arrivals.csv (crossed; 100% ...


mae_15        rmse_15        mape_15  \
dataset  graph                                                       
METR-LA  corr_only     2.720 (0.009)  5.208 (0.018)  7.054 (0.057)   
         express       2.717 (0.004)  5.193 (0.022)  7.009 (0.104)   
         map_only      2.709 (0.006)  5.159 (0.019)  6.956 (0.016)   
         original      2.717 (0.002)  5.201 (0.012)  6.993 (0.059)   
         random        2.724 (0.006)  5.187 (0.030)  7.172 (0.119)   
         same_side     2.710 (0.007)  5.181 (0.011)  7.024 (0.023)   
         virtual_node  2.711 (0.006)  5.177 (0.025)  6.964 (0.029)   
PEMS-BAY corr_only     1.315 (0.003)  2.752 (0.010)  2.767 (0.022)   
         express       1.313 (0.003)  2.754 (0.009)  2.732 (0.037)   
         map_only      1.316 (0.003)  2.761 (0.012)  2.748 (0.018)   
         original      1.317 (0.003)  2.759 (0.015)  2.749 (0.017)   
         random        1.313 (0.004)  2.757 (0.006)  2.730 (0.030)   
         same_side     1.316 (0.003)  2.759 (0.004)  2.748 (0.051)   
         virtual_node  1.318 (0.004)  2.763 (0.015)  2.756 (0.006)   
SHARJAH  corr_only     0.908 (0.002)  1.353 (0.006)  2.256 (0.005)   
         express       0.913 (0.005)  1.363 (0.008)  2.269 (0.032)   
         map_only      0.910 (0.004)  1.355 (0.009)  2.252 (0.010)   
         original      0.910 (0.001)  1.356 (0.002)  2.266 (0.005)   
         random        0.911 (0.002)  1.358 (0.003)  2.259 (0.017)   
         same_side     0.909 (0.001)  1.357 (0.001)  2.261 (0.012)   
         virtual_node  0.908 (0.001)  1.355 (0.003)  2.242 (0.010)   
TOY-C1   original              0.889          1.314          1.755   
         random                0.879          1.300          1.718   

                       mae_15_behind rmse_15_behind mape_15_behind  \
dataset  graph                                                       
METR-LA  corr_only     2.356 (0.005)  3.637 (0.002)  4.322 (0.010)   
         express       2.353 (0.008)  3.642 (0.014)  4.320 (0.015)   
         map_only      2.359 (0.005)  3.636 (0.016)  4.317 (0.014)   
         original      2.359 (0.004)  3.643 (0.014)  4.316 (0.018)   
         random        2.361 (0.013)  3.646 (0.013)  4.326 (0.024)   
         same_side     2.352 (0.005)  3.639 (0.015)  4.316 (0.011)   
         virtual_node  2.346 (0.004)  3.629 (0.012)  4.304 (0.014)   
PEMS-BAY corr_only               nan            nan            nan   
         express                 nan            nan            nan   
         map_only                nan            nan            nan   
         original                nan            nan            nan   
         random                  nan            nan            nan   
         same_side               nan            nan            nan   
         virtual_node            nan            nan            nan   
SHARJAH  corr_only               nan            nan            nan   
         express                 nan            nan            nan   
         map_only                nan            nan            nan   
         original                nan            nan            nan   
         random                  nan            nan            nan   
         same_side               nan            nan            nan   
         virtual_node            nan            nan            nan   
TOY-C1   original                nan            nan            nan   
         random                  nan            nan            nan   

                              mae_30        rmse_30        mape_30  \
dataset  graph                                                       
METR-LA  corr_only     3.117 (0.010)  6.278 (0.021)  8.640 (0.060)   
         express       3.103 (0.012)  6.231 (0.022)  8.478 (0.086)   
         map_only      3.097 (0.003)  6.198 (0.018)  8.462 (0.076)   
         original      3.109 (0.006)  6.262 (0.012)  8.490 (0.069)   
         random        3.118 (0.006)  6.273 (0.030)  8.676 (0.124)   
         same_side     3.105 (0.008)  

Jam recall at 60 min, roads / jams behind the bottleneck: mean (std) over seeds


graph,original,random,map_only,corr_only,same_side,express,virtual_node
dataset,,,,,,,
METR-LA,0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000)
PEMS-BAY,nan,nan,nan,nan,nan,nan,nan
SHARJAH,0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000),0.000 (0.000)
TOY-C1,0.000,0.000,NaN,NaN,NaN,NaN,NaN


## 9. Where everything is saved

All of it is in the shared Drive folder **AdvML_F26_Project_&_Survey_Paper › Project › Coding** (mounted in Colab as `PROJECT_DIR`, printed below). Nothing is kept on the Colab machine, which is why a disconnect costs at most one epoch. One folder per run, three project-wide tables, and the two progress files:

In [ ]:
print('Drive folder : My Drive (or Shared with me) > AdvML_F26_Project_&_Survey_Paper > Project > Coding')
print('in this runtime:', PROJECT_DIR)
print('run folders  :', os.path.join(PROJECT_DIR, RUNS_SUBDIR, '<dataset>', '<graph>', 'seed<k>'), '\n')
layout = [
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/last.pt', 'checkpoint after EVERY epoch: model, optimizer, history, random state; the resume point (about 3.5 MB)', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/best.pt', 'the epoch with the lowest validation MAE; the test uses this one', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/log.csv', 'one row per epoch: train/val MAE, RMSE, MAPE, seconds (the ETA reads it)', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/claim.json', 'who is training this run right now (worker, host, heartbeat); deleted when the run is released', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/train_done.json', 'written when the last epoch is done (the test pass may still be pending)', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/test_predictions.npz', 'the forecasts of best.pt on the test set: pred (samples, 12, sensors) in mph, t_anchor, horizon_min; what steps 16 and 17 read', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/test_by_horizon.csv', 'MAE / RMSE / MAPE at each of the 12 horizons', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/results.json', 'settings + test errors + timing + GPU; "step12": true marks the run as finished', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/<dataset>/<graph>/seed<k>/score.json', 'the step-9 scored row (jam recall / precision, behind-bottleneck subset)', 'metrics.py'),
    (f'{RUNS_SUBDIR}/results.csv', 'one row per finished run: the training side (best epoch, train_hours, sec_per_epoch, gpu, versions)', 'train_gwnet.py'),
    (f'{RUNS_SUBDIR}/scores.csv', 'one row per scored run: every metric (section 8 rebuilds it)', 'metrics.py'),
    (f'{RUNS_SUBDIR}/scores_summary.csv', 'mean (std) over seeds per dataset x graph version (section 8)', 'metrics.py'),
    (f'{RUNS_SUBDIR}/progress_step13.txt, .csv', 'where we are and the ETA of every run, rewritten after every finished run (readable on a phone)', 'this notebook'),
    ('data/raw/toy/SHARJAH_jam_arrivals.csv', "copy of step 11's jam log where metrics.py looks for the behind-bottleneck subset", 'this notebook'),
]
display(pd.DataFrame(layout, columns=['file (under Coding/)', 'what it is', 'written by']))

recs, total_mb = [], 0.0
for rd in sorted(glob.glob(os.path.join(PROJECT_DIR, RUNS_SUBDIR, '*', '*', 'seed*'))):
    files = sorted(f for f in os.listdir(rd) if not f.endswith('.tmp'))
    mb = sum(os.path.getsize(os.path.join(rd, f)) for f in files) / 1e6
    total_mb += mb
    ds, g, sd = rd.split(os.sep)[-3:]
    recs.append(dict(dataset=ds, graph=g, seed=sd, MB=round(mb, 1), files=', '.join(files)))
if recs:
    print(f'\n{len(recs)} run folder(s) in Drive now, {total_mb:.0f} MB in total (about 10 MB per finished run):')
    display(pd.DataFrame(recs))
else:
    print('\nNo run folder yet: the first one appears a few minutes after section 7 starts.')
for f in ['results.csv', 'scores.csv', 'scores_summary.csv', 'progress_step13.txt', 'progress_step13.csv']:
    p = os.path.join(PROJECT_DIR, RUNS_SUBDIR, f)
    print(f'{"OK " if os.path.exists(p) else "-- "} {os.path.relpath(p, PROJECT_DIR)}' + (f'  ({os.path.getsize(p):,} bytes)' if os.path.exists(p) else '  (not written yet)'))

Drive folder : My Drive (or Shared with me) > AdvML_F26_Project_&_Survey_Paper > Project > Coding
in this runtime: /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding
run folders  : /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding/runs/<dataset>/<graph>/seed<k> 



,file (under Coding/),what it is,written by
0,runs/<dataset>/<graph>/seed<k>/last.pt,"checkpoint after EVERY epoch: model, optimizer...",train_gwnet.py
1,runs/<dataset>/<graph>/seed<k>/best.pt,the epoch with the lowest validation MAE; the ...,train_gwnet.py
2,runs/<dataset>/<graph>/seed<k>/log.csv,"one row per epoch: train/val MAE, RMSE, MAPE, ...",train_gwnet.py
3,runs/<dataset>/<graph>/seed<k>/claim.json,"who is training this run right now (worker, ho...",train_gwnet.py
4,runs/<dataset>/<graph>/seed<k>/train_done.json,written when the last epoch is done (the test ...,train_gwnet.py
5,runs/<dataset>/<graph>/seed<k>/test_prediction...,the forecasts of best.pt on the test set: pred...,train_gwnet.py
6,runs/<dataset>/<graph>/seed<k>/test_by_horizon...,MAE / RMSE / MAPE at each of the 12 horizons,train_gwnet.py
7,runs/<dataset>/<graph>/seed<k>/results.json,"settings + test errors + timing + GPU; ""step12...",train_gwnet.py
8,runs/<dataset>/<graph>/seed<k>/score.json,"the step-9 scored row (jam recall / precision,...",metrics.py
9,runs/results.csv,one row per finished run: the training side (b...,train_gwnet.py



70 run folder(s) in Drive now, 6226 MB in total (about 10 MB per finished run):


,dataset,graph,seed,MB,files
0,METR-LA,corr_only,seed0,61.8,"best.pt, last.pt, log.csv, results.json, score..."
1,METR-LA,corr_only,seed1,61.9,"best.pt, last.pt, log.csv, results.json, score..."
2,METR-LA,corr_only,seed2,61.8,"best.pt, last.pt, log.csv, results.json, score..."
3,METR-LA,express,seed0,61.8,"best.pt, last.pt, log.csv, results.json, score..."
4,METR-LA,express,seed1,61.9,"best.pt, last.pt, log.csv, results.json, score..."
...,...,...,...,...,...
65,TOY-C1,check-smooth,seed0,23.3,"results.json, score.json, test_predictions.npz"
66,TOY-C1,corr_only,seed0,4.7,"best.pt, claim.json, last.pt, log.csv"
67,TOY-C1,map_only,seed0,4.7,"best.pt, claim.json, last.pt, log.csv"
68,TOY-C1,original,seed0,59.1,"best.pt, last.pt, log.csv, results.json, score..."


OK  runs/results.csv  (23,406 bytes)
OK  runs/scores.csv  (38,119 bytes)
OK  runs/scores_summary.csv  (7,795 bytes)
OK  runs/progress_step13.txt  (6,615 bytes)
OK  runs/progress_step13.csv  (3,863 bytes)


## Done

Step 13 is done when the counts table in section 8 shows **21 × done** for PEMS-BAY and for SHARJAH, every row **scored**, and (with step 12) 84 done in all. Then:

* **Step 16 (Mohammed), results tables:** `runs/scores.csv` has one row per run (`dataset`, `graph`, `seed`, then every metric: `mae_*` (mean absolute error in mph), `rmse_*`, `mape_*`, `jam_recall_30/60`, `jam_precision_30/60`, the `*_behind` versions, the counts `jams_*` / `warnings_*`, and `subset_source`, which says which file defined "behind the bottleneck" for that dataset). `runs/scores_summary.csv` is the same as mean (std) over seeds per dataset × version. `runs/results.csv` has the training side of each run (`best_epoch`, `train_hours`, `sec_per_epoch`, `gpu`, `epochs`, `patience`), which the setup section of the report needs.
* **Step 17 (Mohammed), paired bootstrap express vs same-side:** for every run, `runs/<dataset>/<graph>/seed<k>/test_predictions.npz` holds the forecasts (`pred` of shape (samples, 12, sensors) in mph, `t_anchor`) and `score.json` the scored row. The paired test needs the per-jam hits: recompute them with `metrics.jam_starts`, `metrics.warnings_at` and `metrics.recall_precision` on the express and same-side forecasts of the same dataset and seed, so each real jam is counted once under both versions.
* **Report:** copy the "across a bottleneck" note from the top of this notebook for PEMS-BAY and Sharjah; the step-12 one covers the toy city and METR-LA. List the GPUs used (`gpu` in `runs/results.csv`; they may differ between runs) and the training hours. If a cut was made (seed 0 only, or fewer epochs), say so and why (section 6's numbers).

**If something goes wrong**

* *`train_gwnet.py` missing or old:* ask Al Zahraa to run section 3 of her step-12 notebook; it writes `code/train_gwnet.py` to Drive. Nothing in this notebook rewrites it.
* *A run shows `in_progress` under YOUR name but nothing is training:* section 6 takes it back after 10 minutes without heartbeat (`RECLAIM_MY_RUNS`). Under ANOTHER name: the module treats the claim as abandoned 30 minutes after its last heartbeat; or that person deletes the folder's `claim.json`.
* *`CONFLICT ... made with other settings`:* a folder holds a checkpoint from different settings (e.g. an earlier test). Delete the folder to retrain, or leave that run to whoever made it.
* *`blocked`:* the graph file for that version is missing; ask Al Zahraa (step 10) and rerun.
* *Section 5 stops on `EPOCHS` / `PATIENCE`:* the step-12 runs in Drive were trained with other values; copy theirs into section 2.
* *`*_behind` columns empty for Sharjah:* `data/raw/toy/SHARJAH_jam_arrivals.csv` is missing; section 5 copies it from `data/raw/sharjah/` when step 11 has run. Rerun section 8 to rescore.
* *Scoring skipped:* the run is saved anyway; section 8 rescores every run.

In [5]:
# ==== Step 13b: rescore all PEMS-BAY and Sharjah runs with the "close to reality" jam check ====
# Paste as ONE new cell at the end of the step-13 notebook and run it (sections 1-3 must have run first,
# so PROJECT_DIR, RUNS_SUBDIR and the metrics module M exist). No GPU needed, nothing is retrained.
#
# New check (SEPA-style, Kralj et al. 2025): a real jam start on road n at time t counts as CAUGHT at horizon H
# if a forecast made H-15 to H minutes before t predicted a speed for the moment t (the jam start)
# within MAX_ERR mph of the real speed. Also reported:
#   jam_mae_H      : average forecast error at the moments jams start (lower = better)
#   false_alarm_H  : share of calm moments (real speed stayed within 5 mph of the last reading) where the
#                    forecast wrongly predicted a drop of >= 12.4 mph  -> guards against "always predict low"
#   persistence    : the same scores for the naive forecast "speed stays as it is now" (a floor to beat)
import os, sys, glob, json, time, importlib
import numpy as np, pandas as pd
try:
    PROJECT_DIR
except NameError:
    raise RuntimeError("Run the two cells of section 1 first (Drive + 'import os, glob, sys...'): they set PROJECT_DIR")
RUNS_SUBDIR = globals().get('RUNS_SUBDIR', 'runs')
sys.path.insert(0, os.path.join(PROJECT_DIR, 'code'))
import metrics as M
M = importlib.reload(M)

MAX_ERR   = 10.0            # mph: "close enough" when the jam hits
H_STEPS   = {30: 6, 60: 12} # minutes -> steps ahead (pred index = steps - 1)
TOL       = M.TOL           # 3 steps = 15 min, the timing slack: forecasts made 45-60 min (or 15-30 min) before the jam
CALM      = 5.0             # mph: a moment is "calm" if the real speed changed less than this
DROP      = M.JAM_DROP      # 12.4 mph: what counts as a predicted drop
DATASETS_13 = ['PEMS-BAY', 'SHARJAH']

def close_scores(pred, D, jams, offset, jam_mask=None):
    """pred (S,12,N); D = test.npz; jams = DataFrame t (raw row), n; offset = raw row of speed[0]."""
    t_index = D['t_index'].astype(int)
    x_last = D['x'][:, -1, :, 0]                                   # last observed speed at each anchor (S,N)
    speed, _ = M.test_series(D)                                     # real speed by row (row r = raw r + offset)
    anchor = {int(a): s for s, a in enumerate(t_index)}
    out = {}
    for mins, h in H_STEPS.items():
        for name in ('model', 'persist'):
            P = pred[:, h - 1, :] if name == 'model' else x_last      # used for false alarms below
            hits, errs, hits_b, = [], [], []
            for i, (t, n) in enumerate(jams[['t', 'n']].values):
                real = speed[t - offset, n]                          # real speed AT the jam start
                if not real > 0:
                    continue
                best = None
                for k in range(h - TOL, h + 1):                      # forecasts made k steps (H-15 .. H min) before it
                    s = anchor.get(int(t - k))
                    if s is None:
                        continue
                    guess = pred[s, k - 1, n] if name == 'model' else x_last[s, n]
                    e = abs(guess - real)
                    best = e if best is None else min(best, e)
                if best is None:
                    continue
                hits.append(best <= MAX_ERR); errs.append(best)
                if jam_mask is not None and jam_mask[i]:
                    hits_b.append(best <= MAX_ERR)
            out[f'{name}_recall_close_{mins}'] = float(np.mean(hits)) if hits else np.nan
            out[f'{name}_jam_mae_{mins}'] = float(np.mean(errs)) if errs else np.nan
            out[f'{name}_recall_close_{mins}_behind'] = float(np.mean(hits_b)) if hits_b else np.nan
            # false alarms: calm moments where the forecast predicted a big drop from the last reading
            rows = t_index + h - offset
            real = speed[rows]                                      # (S,N) real speed at the target moment
            ok = (real > 0) & (x_last > 0)
            calm = ok & (np.abs(real - x_last) < CALM)
            alarm = (x_last - P) >= DROP
            out[f'{name}_false_alarm_{mins}'] = float(alarm[calm].mean()) if calm.any() else np.nan
        out[f'jams_scored'] = len(jams)
    return out

t0 = time.time()
rows = []
data_root = os.path.join(PROJECT_DIR, 'data', 'processed')
raw_toy = os.path.join(PROJECT_DIR, 'data', 'raw', 'toy')
for ds in DATASETS_13:
    dd = os.path.join(data_root, ds)
    D = np.load(os.path.join(dd, 'test.npz'))
    speed, offset = M.test_series(D)
    jams = M.jam_starts(speed); jams['t'] = jams['t'] + offset
    road_mask, jam_mask, src = M.load_subset(dd, jams, 0, raw_toy)
    if road_mask is not None and road_mask.any():
        jam_mask = road_mask[jams['n'].values]
    elif road_mask is not None:
        jam_mask = None                                            # PEMS-BAY: regions.npz marks no road
    run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, RUNS_SUBDIR, ds, '*', 'seed*', 'test_predictions.npz')))
    print(f'{ds}: {len(jams)} real jams, {len(run_dirs)} runs, behind-bottleneck subset: '
          f'{"none" if jam_mask is None else int(jam_mask.sum())} jams')
    for p in run_dirs:
        rd = os.path.dirname(p)
        cfg = json.load(open(os.path.join(rd, 'results.json')))
        P = np.load(p)
        assert np.array_equal(P['t_anchor'], D['t_anchor']), f'anchor mismatch in {rd}'
        r = dict(dataset=ds, graph=cfg['graph'], seed=cfg['seed'])
        r.update(close_scores(P['pred'].astype(np.float32), D, jams, offset, jam_mask))
        rows.append(r)
        print(f'  {cfg["graph"]:>12} seed{cfg["seed"]}: caught@60 {r["model_recall_close_60"]:.3f} '
              f'(persistence {r["persist_recall_close_60"]:.3f}) | false alarms@60 {r["model_false_alarm_60"]:.4f}')

close = pd.DataFrame(rows)
out_csv = os.path.join(PROJECT_DIR, RUNS_SUBDIR, 'scores_close_check.csv')
close.to_csv(out_csv, index=False)
print(f'\nsaved {out_csv}  ({time.time() - t0:.0f} s)')

ORDER = ['original', 'random', 'map_only', 'corr_only', 'same_side', 'express', 'virtual_node']
cols = ['model_recall_close_30', 'model_recall_close_60', 'model_recall_close_60_behind',
        'model_jam_mae_60', 'model_false_alarm_60']
for ds, g in close.groupby('dataset'):
    m = g.groupby('graph')[cols].mean().reindex(ORDER)
    s = g.groupby('graph')[cols].std().reindex(ORDER)
    print(f'\n=== {ds}: mean (std) over seeds | persistence floor: caught@60 '
          f'{g["persist_recall_close_60"].mean():.3f}, jam error@60 {g["persist_jam_mae_60"].mean():.2f} mph')
    display((m.round(3).astype(str) + ' (' + s.round(3).astype(str) + ')'))
    p = g.pivot_table(index='seed', columns='graph', values='model_recall_close_60')
    for other in ['same_side', 'original']:
        d = p['express'] - p[other]
        print(f'  express - {other}: caught@60 {d.mean():+.3f} +- {d.std():.3f}  (per seed: {np.round(d.values, 3).tolist()})')

PEMS-BAY: 7685 real jams, 21 runs, behind-bottleneck subset: none jams
     corr_only seed0: caught@60 0.478 (persistence 0.206) | false alarms@60 0.0026
     corr_only seed1: caught@60 0.545 (persistence 0.206) | false alarms@60 0.0037
     corr_only seed2: caught@60 0.501 (persistence 0.206) | false alarms@60 0.0029
       express seed0: caught@60 0.524 (persistence 0.206) | false alarms@60 0.0027
       express seed1: caught@60 0.530 (persistence 0.206) | false alarms@60 0.0033
       express seed2: caught@60 0.540 (persistence 0.206) | false alarms@60 0.0032
      map_only seed0: caught@60 0.483 (persistence 0.206) | false alarms@60 0.0024
      map_only seed1: caught@60 0.527 (persistence 0.206) | false alarms@60 0.0031
      map_only seed2: caught@60 0.530 (persistence 0.206) | false alarms@60 0.0028
      original seed0: caught@60 0.481 (persistence 0.206) | false alarms@60 0.0024
      original seed1: caught@60 0.547 (persistence 0.206) | false alarms@60 0.0035
      original s

,model_recall_close_30,model_recall_close_60,model_recall_close_60_behind,model_jam_mae_60,model_false_alarm_60
graph,,,,,
original,0.576 (0.02),0.517 (0.033),nan (nan),10.239 (0.495),0.003 (0.001)
random,0.57 (0.03),0.514 (0.031),nan (nan),10.267 (0.49),0.003 (0.001)
map_only,0.572 (0.02),0.513 (0.026),nan (nan),10.278 (0.431),0.003 (0.0)
corr_only,0.569 (0.024),0.508 (0.034),nan (nan),10.381 (0.546),0.003 (0.001)
same_side,0.578 (0.014),0.526 (0.016),nan (nan),10.135 (0.255),0.003 (0.0)
express,0.577 (0.005),0.531 (0.008),nan (nan),9.99 (0.122),0.003 (0.0)
virtual_node,0.598 (0.017),0.546 (0.006),nan (nan),9.808 (0.107),0.003 (0.0)


  express - same_side: caught@60 +0.006 +- 0.020  (per seed: [0.005, -0.014, 0.026])
  express - original: caught@60 +0.014 +- 0.030  (per seed: [0.043, -0.017, 0.017])

=== SHARJAH: mean (std) over seeds | persistence floor: caught@60 0.091, jam error@60 16.62 mph


,model_recall_close_30,model_recall_close_60,model_recall_close_60_behind,model_jam_mae_60,model_false_alarm_60
graph,,,,,
original,0.828 (0.001),0.18 (0.001),0.292 (0.005),14.066 (0.056),0.0 (0.0)
random,0.828 (0.005),0.18 (0.007),0.285 (0.006),14.094 (0.163),0.0 (0.0)
map_only,0.827 (0.001),0.183 (0.002),0.293 (0.007),13.99 (0.083),0.0 (0.0)
corr_only,0.826 (0.004),0.185 (0.001),0.293 (0.002),14.004 (0.04),0.0 (0.0)
same_side,0.827 (0.002),0.181 (0.002),0.28 (0.009),14.084 (0.084),0.0 (0.0)
express,0.828 (0.003),0.182 (0.004),0.284 (0.002),14.07 (0.041),0.0 (0.0)
virtual_node,0.827 (0.007),0.182 (0.001),0.292 (0.002),14.038 (0.068),0.0 (0.0)


  express - same_side: caught@60 +0.000 +- 0.005  (per seed: [0.004, 0.002, -0.005])
  express - original: caught@60 +0.001 +- 0.005  (per seed: [0.003, 0.005, -0.005])


In [3]:
# ==== Step 13c: three cheap checks (no GPU, no training) ====
# Paste as ONE new cell at the end of the step-13 notebook, after the step-13b cell.
# Needs only the two cells of section 1 (Drive + 'import os, glob, sys...') to have run.
#   Check 1  PEMS-BAY "behind the bottleneck": step 5 found NO strong bottleneck on PEMS-BAY (its own rule, fixed
#            before training), so the main subset stays empty on purpose. As a clearly labelled EXPLORATORY
#            extra, we score the 94 roads step 5 flagged as being in a "mild" bottleneck region.
#   Check 2  Do the express links actually cross the bottleneck? (Sharjah: lagoon districts A/B)
#   Check 3  Does the 10 mph "close enough" cutoff drive the ranking? Rescore at 5, 10 and 15 mph.
import os, sys, glob, json, time, pickle, importlib
import numpy as np, pandas as pd
try:
    PROJECT_DIR
except NameError:
    raise RuntimeError("Run the two cells of section 1 first: they set PROJECT_DIR")
RUNS_SUBDIR = globals().get('RUNS_SUBDIR', 'runs')
sys.path.insert(0, os.path.join(PROJECT_DIR, 'code'))
import metrics as M
M = importlib.reload(M)

ORDER = ['original', 'random', 'map_only', 'corr_only', 'same_side', 'express', 'virtual_node']
CUTOFFS = [5.0, 10.0, 15.0]
H = 12                       # 60 min ahead
TOL = M.TOL                  # forecasts made 45-60 min before the jam

def load_pkl(p):
    with open(p, 'rb') as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0); return pickle.load(f, encoding='latin1')

# ------------------------------------------------------------------ check 2
print('=' * 70, '\nCHECK 2: do the express links cross the bottleneck?\n' + '=' * 70)
reg = json.load(open(glob.glob(os.path.join(PROJECT_DIR, 'data', 'raw', '*', 'SHARJAH_regions.json'))[0]))
graphs = os.path.join(PROJECT_DIR, 'data', 'graphs', 'SHARJAH')
orig = load_pkl(glob.glob(os.path.join(PROJECT_DIR, 'data', 'raw', '*', 'SHARJAH_adj_mx.pkl'))[0])
ids = list(orig[0]); A0 = np.asarray(orig[2]) > 0
side = np.array([reg['district'][s] for s in ids])
cross = side[:, None] != side[None, :]
print(f'Sharjah: {(side == "A").sum()} roads on side A, {(side == "B").sum()} on side B; '
      f'original graph has {int(np.triu(A0 & cross, 1).sum())} road links across the lagoon')
rows2 = []
for v in ['express', 'same_side', 'random', 'map_only', 'corr_only']:
    g = load_pkl(os.path.join(graphs, f'{v}_K25.pkl'))
    assert list(g[0]) == ids, f'{v}: sensor order differs from the original graph'
    A = np.asarray(g[2]) > 0
    add = A & ~A0; np.fill_diagonal(add, False)
    i, j = np.nonzero(add)
    rows2.append(dict(version=v, links_added=len(i), across_lagoon=int(cross[i, j].sum()),
                      same_side=int((~cross[i, j]).sum()), links_removed=int((A0 & ~A).sum())))
check2 = pd.DataFrame(rows2).set_index('version')
display(check2)
ex = check2.loc['express']
print(f'-> {ex.across_lagoon} of {ex.links_added} express links cross the lagoon. '
      + ('Most express links stay on one side, so express is close to the same-side control by construction.'
         if ex.across_lagoon < ex.links_added / 2 else 'Most express links do cross the lagoon.'))

# ------------------------------------------------------------------ checks 1 and 3
print('\n' + '=' * 70, '\nCHECKS 1 and 3: rescoring at 5 / 10 / 15 mph (+ PEMS-BAY mild-region subset)\n' + '=' * 70)

def best_errors(get_guess, jams, speed, offset, anchor):
    """For each real jam start: the smallest |forecast - real| among forecasts made 45-60 min before it."""
    out = np.full(len(jams), np.nan)
    for i, (t, n) in enumerate(jams[['t', 'n']].values):
        real = speed[t - offset, n]
        if not real > 0:
            continue
        errs = [abs(get_guess(anchor[t - k], k, n) - real) for k in range(H - TOL, H + 1) if (t - k) in anchor]
        if errs:
            out[i] = min(errs)
    return out

t0 = time.time()
rows = []
for ds in ['PEMS-BAY', 'SHARJAH']:
    dd = os.path.join(PROJECT_DIR, 'data', 'processed', ds)
    D = np.load(os.path.join(dd, 'test.npz'))
    speed, offset = M.test_series(D)
    jams = M.jam_starts(speed); jams['t'] = jams['t'] + offset
    anchor = {int(a): s for s, a in enumerate(D['t_index'].astype(int))}
    x_last = D['x'][:, -1, :, 0]
    subsets = {}
    road_mask, jam_mask, _ = M.load_subset(dd, jams, 0, os.path.join(PROJECT_DIR, 'data', 'raw', 'toy'))
    if jam_mask is not None:
        subsets['crossed (main)'] = jam_mask
    elif road_mask is not None and road_mask.any():
        subsets['behind (main)'] = road_mask[jams['n'].values]
    rp = os.path.join(dd, 'bottleneck', 'regions.npz')
    if os.path.exists(rp):
        r = np.load(rp, allow_pickle=True)
        print(f'{ds}: step-5 regions.npz -> {int(r["behind_bottleneck"].sum())} roads behind a strong bottleneck, '
              f'{int(r["in_mild_region"].sum())} in a mild region')
        if not r['behind_bottleneck'].any() and r['in_mild_region'].any():
            subsets['mild region (exploratory)'] = r['in_mild_region'].astype(bool)[jams['n'].values]
    for name, m in subsets.items():
        print(f'    subset "{name}": {int(m.sum())} of {len(jams)} jams')
    base = best_errors(lambda s, k, n: x_last[s, n], jams, speed, offset, anchor)      # do-nothing forecast
    for p in sorted(glob.glob(os.path.join(PROJECT_DIR, RUNS_SUBDIR, ds, '*', 'seed*', 'test_predictions.npz'))):
        cfg = json.load(open(os.path.join(os.path.dirname(p), 'results.json')))
        pred = np.load(p)['pred'].astype(np.float32)
        e = best_errors(lambda s, k, n: pred[s, k - 1, n], jams, speed, offset, anchor)
        for c in CUTOFFS:
            row = dict(dataset=ds, graph=cfg['graph'], seed=cfg['seed'], cutoff=c,
                       caught_60=np.nanmean(e <= c) if np.isfinite(e).any() else np.nan,
                       do_nothing_60=np.nanmean(base <= c))
            for name, m in subsets.items():
                ok = m & np.isfinite(e)
                row[f'caught_60 | {name}'] = float((e[ok] <= c).mean()) if ok.any() else np.nan
            rows.append(row)
        print(f'  {ds} {cfg["graph"]:>12} seed{cfg["seed"]} done')
res = pd.DataFrame(rows)
res.to_csv(os.path.join(PROJECT_DIR, RUNS_SUBDIR, 'scores_cutoff_check.csv'), index=False)
print(f'saved runs/scores_cutoff_check.csv ({time.time() - t0:.0f} s)')

for ds, g in res.groupby('dataset'):
    print(f'\n=== {ds}: jams caught 60 min ahead, mean over 3 seeds, by cutoff ===')
    cols = [c for c in g.columns if c.startswith('caught_60') and g[c].notna().any()]
    for col in cols:
        t = g.pivot_table(index='graph', columns='cutoff', values=col, aggfunc='mean').reindex(ORDER)
        t.columns = [f'{int(c)} mph' for c in t.columns]
        t['rank @5'] = t['5 mph'].rank(ascending=False).astype(int)
        t['rank @10'] = t['10 mph'].rank(ascending=False).astype(int)
        t['rank @15'] = t['15 mph'].rank(ascending=False).astype(int)
        print(f'\n{col}' + (f'   (do nothing: ' + ', '.join(f'{g[g.cutoff == c].do_nothing_60.mean():.3f}' for c in CUTOFFS) + ')'
                            if col == 'caught_60' else ''))
        display(t.round(3))
    for c in CUTOFFS:
        p = g[g.cutoff == c].pivot_table(index='seed', columns='graph', values='caught_60')
        d = p['express'] - p['same_side']
        print(f'  @{int(c)} mph  express - same_side: {d.mean():+.3f}  per seed {np.round(d.values, 3).tolist()}')

CHECK 2: do the express links cross the bottleneck?
Sharjah: 173 roads on side A, 127 on side B; original graph has 11 road links across the lagoon


,links_added,across_lagoon,same_side,links_removed
version,,,,
express,25,3,22,0
same_side,25,0,25,0
random,25,18,7,0
map_only,25,17,8,0
corr_only,25,0,25,0


-> 3 of 25 express links cross the lagoon. Most express links stay on one side, so express is close to the same-side control by construction.

CHECKS 1 and 3: rescoring at 5 / 10 / 15 mph (+ PEMS-BAY mild-region subset)
PEMS-BAY: step-5 regions.npz -> 0 roads behind a strong bottleneck, 94 in a mild region
    subset "mild region (exploratory)": 2303 of 7685 jams
  PEMS-BAY    corr_only seed0 done
  PEMS-BAY    corr_only seed1 done
  PEMS-BAY    corr_only seed2 done
  PEMS-BAY      express seed0 done
  PEMS-BAY      express seed1 done
  PEMS-BAY      express seed2 done
  PEMS-BAY     map_only seed0 done
  PEMS-BAY     map_only seed1 done
  PEMS-BAY     map_only seed2 done
  PEMS-BAY     original seed0 done
  PEMS-BAY     original seed1 done
  PEMS-BAY     original seed2 done
  PEMS-BAY       random seed0 done
  PEMS-BAY       random seed1 done
  PEMS-BAY       random seed2 done
  PEMS-BAY    same_side seed0 done
  PEMS-BAY    same_side seed1 done
  PEMS-BAY    same_side seed2 done
  PE

,5 mph,10 mph,15 mph,rank @5,rank @10,rank @15
graph,,,,,,
original,0.325,0.517,0.729,5,4,3
random,0.328,0.513,0.725,4,5,6
map_only,0.324,0.513,0.726,6,6,5
corr_only,0.319,0.507,0.723,7,7,7
same_side,0.336,0.525,0.727,3,3,4
express,0.343,0.531,0.739,2,2,2
virtual_node,0.353,0.545,0.748,1,1,1



caught_60 | mild region (exploratory)


,5 mph,10 mph,15 mph,rank @5,rank @10,rank @15
graph,,,,,,
original,0.344,0.550,0.763,6,5,4
random,0.355,0.560,0.769,4,2,2
map_only,0.348,0.546,0.762,5,6,5
corr_only,0.339,0.544,0.758,7,7,7
same_side,0.357,0.554,0.759,2,4,6
express,0.356,0.556,0.769,3,3,3
virtual_node,0.385,0.586,0.784,1,1,1


  @5 mph  express - same_side: +0.007  per seed [0.014, -0.014, 0.022]
  @10 mph  express - same_side: +0.006  per seed [0.005, -0.014, 0.026]
  @15 mph  express - same_side: +0.012  per seed [0.011, -0.002, 0.027]

=== SHARJAH: jams caught 60 min ahead, mean over 3 seeds, by cutoff ===

caught_60   (do nothing: 0.031, 0.091, 0.358)


,5 mph,10 mph,15 mph,rank @5,rank @10,rank @15
graph,,,,,,
original,0.126,0.180,0.507,5,6,4
random,0.126,0.180,0.504,5,7,5
map_only,0.133,0.183,0.511,2,2,1
corr_only,0.129,0.185,0.510,3,1,2
same_side,0.125,0.181,0.507,7,5,3
express,0.129,0.182,0.503,4,4,7
virtual_node,0.136,0.182,0.503,1,3,6



caught_60 | crossed (main)


,5 mph,10 mph,15 mph,rank @5,rank @10,rank @15
graph,,,,,,
original,0.201,0.292,0.559,6,3,1
random,0.197,0.285,0.557,7,5,2
map_only,0.215,0.293,0.555,2,1,4
corr_only,0.213,0.293,0.554,3,1,5
same_side,0.202,0.280,0.556,5,7,3
express,0.206,0.284,0.554,4,6,5
virtual_node,0.221,0.292,0.552,1,3,7


  @5 mph  express - same_side: +0.004  per seed [0.017, 0.003, -0.008]
  @10 mph  express - same_side: +0.000  per seed [0.004, 0.002, -0.005]
  @15 mph  express - same_side: -0.004  per seed [-0.005, -0.006, -0.001]
